# SGCA-VetDerm++ | PyTorch Version — All 8 SOTA Upgrades
### RTX 5090 Ready | Python 3.10 | PyTorch 2.11

| Cell | Content |
|------|---------|
| 1 | Setup, GPU check, config |
| 2 | Dataset loader (auto-detects balanced_data/) |
| 3 | SGCA model in PyTorch + baseline eval |
| 4 | U1: Kendall-Gal uncertainty weighting |
| 5 | U2: MC-Dropout epistemic uncertainty |
| 6 | U3: Hierarchical prototypical loss |
| 7 | U4: Focal + Label Smoothing loss |
| 8 | U5: Conformal prediction sets |
| 9 | V6: PCGrad gradient surgery |
| 10 | V7: Species-Conditioned MoE |
| 11 | V8: SSM benchmark |
| 12 | Final ablation table + chart |

> **Kernel:** select `/mnt/d/Projects/RFantibody/.venv/bin/python` in Jupyter

## Cell 1 — Setup & GPU Verification
> ✏️ Edit `MODEL_PATH` and `DATA_DIR` only

In [9]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
x = torch.randn(4, 4, device='cuda', requires_grad=True)
loss = (x @ x).sum()
loss.backward()
print("✓ backward pass works")

CUDA: True
GPU: NVIDIA GeForce RTX 5090 D v2
✓ backward pass works


In [10]:
import os
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/cuda-12.8/targets/x86_64-linux/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

import ctypes
ctypes.CDLL("/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcupti.so.12")

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")

PyTorch: 2.11.0a0+git578fcce
CUDA: True
GPU: NVIDIA GeForce RTX 5090 D v2


In [11]:
import sys, torch
print(sys.executable)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

/mnt/d/Projects/RFantibody/.venv/bin/python
2.11.0a0+git578fcce
True
NVIDIA GeForce RTX 5090 D v2


In [12]:
# ══════════════════════════════════════════════
# EDIT ONLY THESE TWO LINES
# ══════════════════════════════════════════════
MODEL_PATH  = "VetSGCA_sgca_Final.keras"   # will convert weights automatically
DATA_DIR    = "balanced_data"
IMG_SIZE    = 299
BATCH_SIZE  = 16                            # reduce to 8 if OOM
EPOCHS      = 10
RESULTS_DIR = "sgca_results_pt"
# ══════════════════════════════════════════════

import os, sys, math, time, warnings, copy
os.makedirs(RESULTS_DIR, exist_ok=True)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

# ── GPU ──────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"PyTorch : {torch.__version__}")
    # Enable TF32 for RTX 5090 — faster matmul with minimal precision loss
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
else:
    print("⚠  No GPU — running on CPU")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def rpath(f): return os.path.join(RESULTS_DIR, f)
all_results = []
print("\n✓ Setup complete")

Device  : cuda
GPU     : NVIDIA GeForce RTX 5090 D v2
VRAM    : 25.6 GB
PyTorch : 2.11.0a0+git578fcce

✓ Setup complete


## Cell 2 — Dataset
Auto-detects your `balanced_data/` folder structure. No manual label mapping needed.

In [13]:
# ── Discover structure ───────────────────────────────────────
SPECIES_NAMES = sorted([d for d in os.listdir(DATA_DIR)
                         if os.path.isdir(os.path.join(DATA_DIR, d))])
print(f"Species: {SPECIES_NAMES}")

all_paths, all_species, all_diseases = [], [], []
for sp in SPECIES_NAMES:
    sp_dir = os.path.join(DATA_DIR, sp)
    for dis in sorted(os.listdir(sp_dir)):
        dis_dir = os.path.join(sp_dir, dis)
        if not os.path.isdir(dis_dir): continue
        for f in os.listdir(dis_dir):
            if f.lower().endswith((".jpg",".jpeg",".png",".bmp",".tiff")):
                all_paths.append(os.path.join(dis_dir, f))
                all_species.append(sp)
                all_diseases.append(dis)

print(f"Total images: {len(all_paths)}")

sp_enc  = LabelEncoder().fit(all_species)
dis_enc = LabelEncoder().fit(all_diseases)
sp_labels  = sp_enc.transform(all_species)
dis_labels = dis_enc.transform(all_diseases)

SPECIES_NAMES = list(sp_enc.classes_)
DISEASE_NAMES = list(dis_enc.classes_)
N_SPECIES = len(SPECIES_NAMES)
N_DISEASES = len(DISEASE_NAMES)
print(f"Species  ({N_SPECIES}): {SPECIES_NAMES}")
print(f"Diseases ({N_DISEASES}): {DISEASE_NAMES[:5]} ...")

# ── Stratified 70/15/15 split ─────────────────────────────────
idx = np.arange(len(all_paths))
idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, stratify=dis_labels, random_state=SEED)
idx_val, idx_te = train_test_split(idx_tmp, test_size=0.50,
                                    stratify=dis_labels[idx_tmp], random_state=SEED)
print(f"Split — train:{len(idx_tr)}  val:{len(idx_val)}  test:{len(idx_te)}")

# ── PyTorch Dataset ───────────────────────────────────────────
train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class VetDermDataset(Dataset):
    def __init__(self, indices, transform):
        self.indices = indices
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        try:
            img = Image.open(all_paths[idx]).convert("RGB")
        except Exception:
            img = Image.fromarray(np.zeros((IMG_SIZE,IMG_SIZE,3),dtype=np.uint8))
        return (self.transform(img),
                torch.tensor(sp_labels[idx],  dtype=torch.long),
                torch.tensor(dis_labels[idx], dtype=torch.long))

train_ds  = VetDermDataset(idx_tr,  train_tf)
val_ds    = VetDermDataset(idx_val, val_tf)
test_ds   = VetDermDataset(idx_te,  val_tf)

train_dl  = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)
val_dl    = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)
test_dl   = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)
print("\n✓ DataLoaders ready")

Species: ['Cat', 'Cattles', 'Dog']
Total images: 10133
Species  (3): ['Cat', 'Cattles', 'Dog']
Diseases (19): ['Cat_normal', 'Dog_normal', 'Ear Mites in Cat', 'Eye Infection in Cat', 'Eye Infection in Dog'] ...
Split — train:7093  val:1520  test:1520

✓ DataLoaders ready


## Cell 3 — SGCA Model in PyTorch + Baseline Evaluation

Rebuilds the SGCA architecture natively in PyTorch using `timm` EfficientNetV2S backbone.
Loads weights from your `.keras` file automatically via a weight-conversion helper.

> ⚠️ If weight conversion fails, the model trains from scratch (ImageNet pretrained).
> Results will be slightly lower initially but will match after fine-tuning.

In [14]:
# ── SGCA PyTorch Architecture ────────────────────────────────
class SpeciesGatedCrossAttention(nn.Module):
    """
    PyTorch reimplementation of the Species-Gated Cross-Attention block.
    Species logits gate the disease feature extraction pathway.
    """
    def __init__(self, feat_dim, n_species, n_diseases, dropout=0.3):
        super().__init__()
        self.sp_head  = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.GELU(), nn.BatchNorm1d(256),
            nn.Dropout(dropout), nn.Linear(256, n_species)
        )
        # Species-conditioned gating
        self.gate = nn.Sequential(
            nn.Linear(n_species, feat_dim), nn.Sigmoid()
        )
        self.dis_head = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.GELU(), nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, 256), nn.GELU(), nn.BatchNorm1d(256),
            nn.Dropout(dropout),
            nn.Linear(256, n_diseases)
        )

    def forward(self, x):
        sp_logits  = self.sp_head(x)                          # (B, n_sp)
        gate       = self.gate(sp_logits.detach())            # (B, feat_dim)
        x_gated    = x * gate                                 # species-gated features
        dis_logits = self.dis_head(x_gated)
        return sp_logits, dis_logits, x_gated                 # also return embedding


class SGCAVetDerm(nn.Module):
    """Full SGCA-VetDerm model with EfficientNetV2S backbone."""
    def __init__(self, n_species, n_diseases, dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            "tf_efficientnetv2_s",
            pretrained=pretrained,
            num_classes=0,          # remove classifier head
            global_pool="avg"       # global average pooling
        )
        feat_dim = self.backbone.num_features   # 1280 for EfficientNetV2S
        self.head = SpeciesGatedCrossAttention(feat_dim, n_species, n_diseases, dropout)

    def forward(self, x, return_embedding=False):
        feats = self.backbone(x)                              # (B, 1280)
        sp_logits, dis_logits, embedding = self.head(feats)
        if return_embedding:
            return sp_logits, dis_logits, embedding
        return sp_logits, dis_logits


# ── Evaluation helper ─────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device=DEVICE):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, sp_lbs, dis_lbs in loader:
        imgs = imgs.to(device)
        sp_out, dis_out = model(imgs)
        preds = dis_out.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(dis_lbs.numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return (accuracy_score(labels, preds),
            f1_score(labels, preds, average="macro", zero_division=0),
            preds, labels)


# ── Build model ───────────────────────────────────────────────
print("Building SGCA-VetDerm (PyTorch)...")
model = SGCAVetDerm(N_SPECIES, N_DISEASES, pretrained=True).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

# ── Try to load weights from .keras file ─────────────────────
WEIGHTS_LOADED = False
if os.path.exists(MODEL_PATH):
    print(f"\nAttempting weight transfer from {MODEL_PATH}...")
    try:
        # Try TF→PT weight conversion via keras
        import tensorflow as tf
        keras_model = tf.keras.models.load_model(MODEL_PATH)
        print("  Keras model loaded — weight mapping in progress...")
        # Map backbone weights layer by layer where shapes match
        keras_weights = {w.name: w.numpy() for w in keras_model.weights}
        matched = 0
        for name, param in model.named_parameters():
            # Try direct name match after normalization
            for kname, kw in keras_weights.items():
                if (param.data.shape == torch.tensor(kw).shape and
                        any(n in kname for n in name.split(".")[-2:])):
                    param.data = torch.tensor(kw).to(DEVICE)
                    matched += 1
                    break
        print(f"  Matched {matched} parameter tensors from keras model")
        WEIGHTS_LOADED = matched > 10
        del keras_model
    except Exception as e:
        print(f"  Weight transfer skipped: {e}")
        print("  Using ImageNet pretrained weights (fine-tuning will recover performance)")
else:
    print(f"\n⚠  {MODEL_PATH} not found — using ImageNet pretrained backbone")
    print("   Copy your .keras file to the same folder as this notebook")

# ── Baseline eval ─────────────────────────────────────────────
print("\nRunning baseline evaluation...")
acc_b, f1_b, preds_base, labels_base = evaluate(model, test_dl)
print(f"\n{'='*45}")
print(f"  BASELINE")
print(f"{'='*45}")
print(f"  Accuracy  : {acc_b:.4f}")
print(f"  Macro-F1  : {f1_b:.4f}")
print(f"  Weights   : {'Transferred from .keras' if WEIGHTS_LOADED else 'ImageNet pretrained'}")
print(f"{'='*45}")

# Per-species breakdown
_, _, sp_preds_b, _ = evaluate(model, test_dl)
sp_labels_test = np.array([sp_labels[i] for i in idx_te])
for sp_id, sp_name in enumerate(SPECIES_NAMES):
    mask = sp_labels_test == sp_id
    if mask.sum() > 0:
        sp_acc = accuracy_score(labels_base[mask], preds_base[mask])
        print(f"  {sp_name:10s}: {sp_acc:.4f}  ({mask.sum()} samples)")

all_results.append({"Model":"SGCA Baseline","Accuracy":round(acc_b,4),"Macro-F1":round(f1_b,4)})
print("\n✓ Baseline complete — proceed to Cell 4")

Building SGCA-VetDerm (PyTorch)...
Parameters: 21,305,446

Attempting weight transfer from VetSGCA_sgca_Final.keras...
  Weight transfer skipped: No module named 'tensorflow'
  Using ImageNet pretrained weights (fine-tuning will recover performance)

Running baseline evaluation...

  BASELINE
  Accuracy  : 0.0500
  Macro-F1  : 0.0165
  Weights   : ImageNet pretrained
  Cat       : 0.0171  (468 samples)
  Cattles   : 0.0645  (496 samples)
  Dog       : 0.0647  (556 samples)

✓ Baseline complete — proceed to Cell 4


## Cell 4 — U1: Kendall-Gal Uncertainty-Weighted Loss
$$L = e^{-s_s}L_{sp} + s_s + e^{-s_d}L_{dis} + s_d$$
Learnable $s_s, s_d$ replace fixed `loss_weights`. Model discovers optimal ratio automatically.

In [15]:
class UncertaintyWeightedTrainer:
    """Wraps SGCAVetDerm and adds two learnable log-variance scalars."""
    def __init__(self, base_model, lr=3e-5, wd=1e-4):
        self.model    = base_model
        self.log_s_sp  = nn.Parameter(torch.zeros(1, device=DEVICE))
        self.log_s_dis = nn.Parameter(torch.zeros(1, device=DEVICE))
        self.optimizer = torch.optim.AdamW(
            list(base_model.parameters()) + [self.log_s_sp, self.log_s_dis],
            lr=lr, weight_decay=wd
        )
        self.ce = nn.CrossEntropyLoss()

    def train_epoch(self, loader):
        self.model.train()
        tot, n = 0., 0
        sig_sp_hist, sig_dis_hist = [], []
        for imgs, sp_lbs, dis_lbs in loader:
            imgs    = imgs.to(DEVICE)
            sp_lbs  = sp_lbs.to(DEVICE)
            dis_lbs = dis_lbs.to(DEVICE)
            self.optimizer.zero_grad()
            sp_out, dis_out = self.model(imgs)
            L_sp  = self.ce(sp_out,  sp_lbs)
            L_dis = self.ce(dis_out, dis_lbs)
            # Kendall-Gal uncertainty weighting
            loss  = (torch.exp(-self.log_s_sp)  * L_sp  + self.log_s_sp +
                     torch.exp(-self.log_s_dis) * L_dis + self.log_s_dis)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            tot += loss.item(); n += 1
            sig_sp_hist.append(torch.exp(self.log_s_sp/2).item())
            sig_dis_hist.append(torch.exp(self.log_s_dis/2).item())
        return tot/n, np.mean(sig_sp_hist), np.mean(sig_dis_hist)


# ── Train ─────────────────────────────────────────────────────
model_u1 = copy.deepcopy(model)
trainer_u1 = UncertaintyWeightedTrainer(model_u1)

sigma_sp_hist, sigma_dis_hist = [], []
best_f1, patience, best_wts = 0., 0, None
print("Training U1: Kendall-Gal uncertainty weighting...")

for ep in range(EPOCHS):
    loss, sig_sp, sig_dis = trainer_u1.train_epoch(train_dl)
    acc_v, f1_v, _, _ = evaluate(model_u1, val_dl)
    sigma_sp_hist.append(sig_sp); sigma_dis_hist.append(sig_dis)
    print(f"  Ep {ep+1:2d} | loss={loss:.4f} | σ_sp={sig_sp:.4f} σ_dis={sig_dis:.4f} | val_f1={f1_v:.4f}")
    if f1_v > best_f1:
        best_f1 = f1_v; patience = 0
        best_wts = copy.deepcopy(model_u1.state_dict())
    else:
        patience += 1
        if patience >= 4: print("  Early stopping"); break

model_u1.load_state_dict(best_wts)
acc_u1, f1_u1, _, _ = evaluate(model_u1, test_dl)
sig_sp_final  = float(torch.exp(trainer_u1.log_s_sp/2).detach().cpu())
sig_dis_final = float(torch.exp(trainer_u1.log_s_dis/2).detach().cpu())

print(f"\n  Learned σ_species  = {sig_sp_final:.4f}")
print(f"  Learned σ_disease  = {sig_dis_final:.4f}")
print(f"  Effective ratio sp:dis = {1/sig_sp_final**2:.3f} : {1/sig_dis_final**2:.3f}")
print(f"  U1 Accuracy: {acc_u1:.4f}  Macro-F1: {f1_u1:.4f}")
all_results.append({"Model":"U1: Kendall-Gal","Accuracy":round(acc_u1,4),"Macro-F1":round(f1_u1,4)})

# ── Plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(sigma_sp_hist,  lw=2, label="σ_species",  color="#2E86C1")
ax.plot(sigma_dis_hist, lw=2, label="σ_disease",  color="#CB4335")
ax.set_title("U1: Learned Task Noise σ — Kendall-Gal\n(lower σ = higher task weight)", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("σ"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(rpath("u1_sigma.png"), dpi=600, bbox_inches="tight"); plt.show()
torch.save(model_u1.state_dict(), rpath("model_u1.pt"))
print("\n✓ Saved: u1_sigma.png, model_u1.pt")

Training U1: Kendall-Gal uncertainty weighting...
  Ep  1 | loss=2.4556 | σ_sp=0.9967 σ_dis=1.0026 | val_f1=0.7055
  Ep  2 | loss=1.1377 | σ_sp=0.9888 σ_dis=1.0048 | val_f1=0.8245
  Ep  3 | loss=0.6500 | σ_sp=0.9814 σ_dis=1.0028 | val_f1=0.9104
  Ep  4 | loss=0.3672 | σ_sp=0.9744 σ_dis=0.9972 | val_f1=0.9344
  Ep  5 | loss=0.1902 | σ_sp=0.9676 σ_dis=0.9899 | val_f1=0.9522
  Ep  6 | loss=0.0799 | σ_sp=0.9609 σ_dis=0.9823 | val_f1=0.9576
  Ep  7 | loss=0.0054 | σ_sp=0.9543 σ_dis=0.9749 | val_f1=0.9669
  Ep  8 | loss=-0.0576 | σ_sp=0.9479 σ_dis=0.9678 | val_f1=0.9695
  Ep  9 | loss=-0.1112 | σ_sp=0.9415 σ_dis=0.9608 | val_f1=0.9703
  Ep 10 | loss=-0.1512 | σ_sp=0.9352 σ_dis=0.9541 | val_f1=0.9677

  Learned σ_species  = 0.9321
  Learned σ_disease  = 0.9508
  Effective ratio sp:dis = 1.151 : 1.106
  U1 Accuracy: 0.9763  Macro-F1: 0.9763

✓ Saved: u1_sigma.png, model_u1.pt


## Cell 5 — U2: Monte Carlo Dropout
Run T=30 forward passes with dropout **active** at inference time.
- **Epistemic** uncertainty: $MI = H_{pred} - H_{aleatoric}$ (model doesn't know)
- **Aleatoric** uncertainty: irreducible image ambiguity

In [16]:
model = model_u1
T_PASSES = 30
print(f"Running {T_PASSES} MC-Dropout passes...")

def mc_predict(model, loader, T, device=DEVICE):
    """Run T stochastic forward passes. Returns (T, N, C) probability array."""
    model.train()   # keeps dropout ACTIVE
    all_pass = []
    with torch.no_grad():
        for t in range(T):
            preds_t = []
            for imgs, _, _ in loader:
                imgs = imgs.to(device)
                _, dis_out = model(imgs)
                preds_t.append(F.softmax(dis_out, dim=1).cpu().numpy())
            all_pass.append(np.concatenate(preds_t, axis=0))
            if (t+1) % 5 == 0: print(f"  Pass {t+1}/{T}")
    return np.array(all_pass)   # (T, N, C)

all_pass_probs = mc_predict(model, test_dl, T_PASSES)
probs_mc  = all_pass_probs.mean(axis=0)                     # (N, C)
preds_mc  = probs_mc.argmax(axis=1)

# Ground truth labels for test set
labels_test = np.array([dis_labels[i] for i in idx_te])
sp_labels_test = np.array([sp_labels[i] for i in idx_te])

eps = 1e-8
H_pred  = -(probs_mc * np.log(probs_mc + eps)).sum(axis=1)
H_pass  = -(all_pass_probs * np.log(all_pass_probs + eps)).sum(axis=2)
H_alea  = H_pass.mean(axis=0)
MI      = np.maximum(H_pred - H_alea, 0)
correct = preds_mc == labels_test

acc_mc = accuracy_score(labels_test, preds_mc)
f1_mc  = f1_score(labels_test, preds_mc, average="macro", zero_division=0)
print(f"\n  MC-Dropout Accuracy: {acc_mc:.4f}  Macro-F1: {f1_mc:.4f}")
print(f"  Mean MI correct   : {MI[correct].mean():.4f}")
print(f"  Mean MI incorrect : {MI[~correct].mean():.4f}")
all_results.append({"Model":f"U2: MC-Dropout T={T_PASSES}","Accuracy":round(acc_mc,4),"Macro-F1":round(f1_mc,4)})

# Store for Cell 8
probs_mc_test = probs_mc

# ── Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, vals, title, col in [
    (axes[0], H_pred, "Total Uncertainty\n(Predictive Entropy)", "#2E86C1"),
    (axes[1], MI,     "Epistemic\n(Mutual Information)",          "#CB4335"),
    (axes[2], H_alea, "Aleatoric\n(Irreducible)",                "#1E8449"),
]:
    ax.hist(vals[correct],  bins=40, alpha=0.65, color=col,    label="Correct",   density=True)
    ax.hist(vals[~correct], bins=40, alpha=0.65, color="gray", label="Incorrect", density=True)
    ax.set_title(title, fontweight="bold"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.suptitle(f"U2: MC-Dropout Uncertainty Decomposition (T={T_PASSES})", fontweight="bold")
plt.tight_layout()
plt.savefig(rpath("u2_mc_uncertainty.png"), dpi=600, bbox_inches="tight"); plt.show()

# Per-class epistemic
per_class_mi = [MI[labels_test==c].mean() if (labels_test==c).sum()>0 else 0
                for c in range(N_DISEASES)]
fig, ax = plt.subplots(figsize=(14,4))
colors  = ["#CB4335" if v > np.mean(per_class_mi) else "#2E86C1" for v in per_class_mi]
ax.bar(range(N_DISEASES), per_class_mi, color=colors)
ax.set_xticks(range(N_DISEASES))
ax.set_xticklabels(DISEASE_NAMES, rotation=45, ha="right", fontsize=8)
ax.axhline(np.mean(per_class_mi), color="black", ls="--", lw=1.5, label="Mean MI")
ax.set_title("U2: Per-Class Epistemic Uncertainty (red = most uncertain)", fontweight="bold")
ax.legend(); plt.tight_layout()
plt.savefig(rpath("u2_per_class.png"), dpi=600, bbox_inches="tight"); plt.show()
print("\n✓ Saved: u2_mc_uncertainty.png, u2_per_class.png")

Running 30 MC-Dropout passes...
  Pass 5/30
  Pass 10/30
  Pass 15/30
  Pass 20/30
  Pass 25/30
  Pass 30/30

  MC-Dropout Accuracy: 0.9526  Macro-F1: 0.9530
  Mean MI correct   : 0.0104
  Mean MI incorrect : 0.0627

✓ Saved: u2_mc_uncertainty.png, u2_per_class.png


## Cell 6 — U3: Hierarchical Prototypical Loss
$$L_{proto} = \alpha L_{pull} + \beta L_{push}$$
Pull same-species-disease embeddings together. Push cross-species same-disease embeddings apart.

In [17]:
# Auto-build cross-species groups from disease name keywords
CROSS_GROUPS = []
for kw in ["normal","ringworm","scabies","mange","eye","allergy","dermatitis","infection"]:
    grp = [i for i, n in enumerate(DISEASE_NAMES) if kw.lower() in n.lower()]
    if len(grp) >= 2:
        CROSS_GROUPS.append(grp)
if not CROSS_GROUPS:
    CROSS_GROUPS = [[0, N_DISEASES//3]]
print(f"Cross-species groups: {len(CROSS_GROUPS)}")


def proto_loss_fn(emb, y_sp, y_dis, margin=1.0, alpha=0.5):
    """Compute L_pull + L_push over embedding batch."""
    pulls, pushes = [], []
    for sp in range(N_SPECIES):
        for dis in range(N_DISEASES):
            mask = (y_sp == sp) & (y_dis == dis)
            if mask.sum() < 2: continue
            z = emb[mask]
            proto = z.mean(dim=0, keepdim=True)
            pulls.append(((z - proto)**2).sum(dim=1).mean())
    for grp in CROSS_GROUPS:
        protos = []
        for d in grp:
            mask = y_dis == d
            if mask.sum() < 1: continue
            protos.append(emb[mask].mean(dim=0))
        for i in range(len(protos)):
            for j in range(i+1, len(protos)):
                d2 = ((protos[i] - protos[j])**2).sum()
                pushes.append(F.relu(margin - d2))
    L_pull = torch.stack(pulls).mean()  if pulls  else torch.tensor(0., device=DEVICE)
    L_push = torch.stack(pushes).mean() if pushes else torch.tensor(0., device=DEVICE)
    return alpha * L_pull + (1-alpha) * L_push


model_u3 = copy.deepcopy(model)
optimizer_u3 = torch.optim.AdamW(model_u3.parameters(), lr=3e-5, weight_decay=1e-4)
ce = nn.CrossEntropyLoss()
lam = 0.1

print("Training U3: Prototypical Loss...")
best_f1_u3, patience_u3, best_wts_u3 = 0., 0, None
proto_hist = []

for ep in range(EPOCHS):
    model_u3.train()
    ep_loss, ep_proto, n = 0., 0., 0
    for imgs, sp_lbs, dis_lbs in train_dl:
        imgs    = imgs.to(DEVICE)
        sp_lbs  = sp_lbs.to(DEVICE)
        dis_lbs = dis_lbs.to(DEVICE)
        optimizer_u3.zero_grad()
        sp_out, dis_out, emb = model_u3(imgs, return_embedding=True)
        loss_ce = ce(sp_out, sp_lbs) + 2*ce(dis_out, dis_lbs)
        loss_p  = proto_loss_fn(emb, sp_lbs, dis_lbs)
        loss    = loss_ce + lam * loss_p
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_u3.parameters(), 1.0)
        optimizer_u3.step()
        ep_loss += loss.item(); ep_proto += loss_p.item(); n += 1
    proto_hist.append(ep_proto/n)
    acc_v, f1_v, _, _ = evaluate(model_u3, val_dl)
    print(f"  Ep {ep+1:2d} | L_proto={ep_proto/n:.4f} | val_f1={f1_v:.4f}")
    if f1_v > best_f1_u3:
        best_f1_u3 = f1_v; patience_u3 = 0
        best_wts_u3 = copy.deepcopy(model_u3.state_dict())
    else:
        patience_u3 += 1
        if patience_u3 >= 4: print("  Early stopping"); break

model_u3.load_state_dict(best_wts_u3)
acc_u3, f1_u3, _, _ = evaluate(model_u3, test_dl)
print(f"\n  U3 Accuracy: {acc_u3:.4f}  Macro-F1: {f1_u3:.4f}")
all_results.append({"Model":"U3: Prototypical","Accuracy":round(acc_u3,4),"Macro-F1":round(f1_u3,4)})

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(proto_hist, lw=2, color="#CB4335")
ax.set_title("U3: Prototypical Loss Trajectory", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("L_proto"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(rpath("u3_proto.png"), dpi=600, bbox_inches="tight"); plt.show()
print("\n✓ Saved: u3_proto.png")

Cross-species groups: 6
Training U3: Prototypical Loss...
  Ep  1 | L_proto=2.9701 | val_f1=0.9666
  Ep  2 | L_proto=0.4052 | val_f1=0.9662
  Ep  3 | L_proto=0.2952 | val_f1=0.9690
  Ep  4 | L_proto=0.2485 | val_f1=0.9732
  Ep  5 | L_proto=0.2078 | val_f1=0.9729
  Ep  6 | L_proto=0.1800 | val_f1=0.9744
  Ep  7 | L_proto=0.1646 | val_f1=0.9772
  Ep  8 | L_proto=0.1496 | val_f1=0.9767
  Ep  9 | L_proto=0.1432 | val_f1=0.9748
  Ep 10 | L_proto=0.1235 | val_f1=0.9718

  U3 Accuracy: 0.9743  Macro-F1: 0.9742

✓ Saved: u3_proto.png


## Cell 7 — U4: Focal Loss + Label Smoothing
$$L_{FLS} = -(1-p_y)^\gamma \sum_c q_c \log p_c, \quad q_c = (1-\varepsilon)\mathbf{1}[c=y] + \varepsilon/K$$
γ=2 down-weights easy examples. ε=0.1 prevents overconfident saturation.

In [18]:
class FocalLabelSmoothing(nn.Module):
    def __init__(self, n_classes, gamma=2.0, epsilon=0.1):
        super().__init__()
        self.K = n_classes; self.gamma = gamma; self.eps = epsilon

    def forward(self, logits, targets):
        probs   = F.softmax(logits, dim=1)                      # (B, K)
        y_oh    = F.one_hot(targets, self.K).float()            # (B, K)
        q       = (1-self.eps)*y_oh + self.eps/self.K           # smoothed targets
        p_true  = (probs * y_oh).sum(dim=1)                     # (B,)
        focal_w = (1.0 - p_true).pow(self.gamma)               # (B,)
        log_p   = torch.log(probs + 1e-8)                       # (B, K)
        ls_ce   = -(q * log_p).sum(dim=1)                       # (B,)
        return (focal_w * ls_ce).mean()


model_u4 = copy.deepcopy(model)
optimizer_u4 = torch.optim.AdamW(model_u4.parameters(), lr=3e-5, weight_decay=1e-4)
fls_loss = FocalLabelSmoothing(N_DISEASES, gamma=2.0, epsilon=0.1)
ce_sp    = nn.CrossEntropyLoss()

print("Training U4: Focal + Label Smoothing...")
best_f1_u4, patience_u4, best_wts_u4 = 0., 0, None

for ep in range(EPOCHS):
    model_u4.train()
    ep_loss, n = 0., 0
    for imgs, sp_lbs, dis_lbs in train_dl:
        imgs    = imgs.to(DEVICE)
        sp_lbs  = sp_lbs.to(DEVICE)
        dis_lbs = dis_lbs.to(DEVICE)
        optimizer_u4.zero_grad()
        sp_out, dis_out = model_u4(imgs)
        loss = ce_sp(sp_out, sp_lbs) + 2*fls_loss(dis_out, dis_lbs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_u4.parameters(), 1.0)
        optimizer_u4.step()
        ep_loss += loss.item(); n += 1
    acc_v, f1_v, _, _ = evaluate(model_u4, val_dl)
    print(f"  Ep {ep+1:2d} | loss={ep_loss/n:.4f} | val_f1={f1_v:.4f}")
    if f1_v > best_f1_u4:
        best_f1_u4 = f1_v; patience_u4 = 0
        best_wts_u4 = copy.deepcopy(model_u4.state_dict())
    else:
        patience_u4 += 1
        if patience_u4 >= 4: print("  Early stopping"); break

model_u4.load_state_dict(best_wts_u4)
acc_u4, f1_u4, preds_u4, _ = evaluate(model_u4, test_dl)
print(f"\n  U4 Accuracy: {acc_u4:.4f}  Macro-F1: {f1_u4:.4f}")
all_results.append({"Model":"U4: Focal+LS","Accuracy":round(acc_u4,4),"Macro-F1":round(f1_u4,4)})

# Per-class F1 chart
f1_b_cls = f1_score(labels_test, preds_base, average=None, zero_division=0)
f1_u4_cls = f1_score(labels_test, preds_u4,  average=None, zero_division=0)
delta = f1_u4_cls - f1_b_cls
fig, axes = plt.subplots(2, 1, figsize=(14,8))
x = np.arange(N_DISEASES); w = 0.35
axes[0].bar(x-w/2, f1_b_cls,  w, label="Baseline CE",   color="#BDC3C7")
axes[0].bar(x+w/2, f1_u4_cls, w, label="Focal+LS γ=2",  color="#2E86C1")
axes[0].set_xticks(x); axes[0].set_xticklabels(DISEASE_NAMES, rotation=45, ha="right", fontsize=8)
axes[0].set_title("U4: Per-Class F1 Baseline vs Focal+LS", fontweight="bold"); axes[0].legend()
axes[1].bar(x, delta, color=["#1E8449" if d>0 else "#CB4335" for d in delta])
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xticks(x); axes[1].set_xticklabels(DISEASE_NAMES, rotation=45, ha="right", fontsize=8)
axes[1].set_title("ΔF1 per class (green=improved)", fontweight="bold")
plt.tight_layout()
plt.savefig(rpath("u4_focal_ls.png"), dpi=600, bbox_inches="tight"); plt.show()
print("\n✓ Saved: u4_focal_ls.png")

Training U4: Focal + Label Smoothing...
  Ep  1 | loss=0.0720 | val_f1=0.9709
  Ep  2 | loss=0.0796 | val_f1=0.9700
  Ep  3 | loss=0.0557 | val_f1=0.9748
  Ep  4 | loss=0.0432 | val_f1=0.9788
  Ep  5 | loss=0.0453 | val_f1=0.9747
  Ep  6 | loss=0.0390 | val_f1=0.9734
  Ep  7 | loss=0.0419 | val_f1=0.9747
  Ep  8 | loss=0.0338 | val_f1=0.9722
  Early stopping

  U4 Accuracy: 0.9711  Macro-F1: 0.9710

✓ Saved: u4_focal_ls.png


## Cell 8 — U5: Conformal Prediction
$$P(Y \in C(X)) \geq 1-\alpha \quad \text{(distribution-free, no assumptions)}$$
No retraining needed. Works on any model.

In [19]:
# Get val probabilities for calibration
print("Getting val set probabilities for calibration...")

@torch.no_grad()
def get_probs(model, loader, device=DEVICE):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, _, dis_lbs in loader:
        imgs = imgs.to(device)
        _, dis_out = model(imgs)
        all_probs.append(F.softmax(dis_out, dim=1).cpu().numpy())
        all_labels.append(dis_lbs.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

# Use MC probs if available, else standard
try:
    probs_cal = probs_mc_val
    probs_tst = probs_mc_test
    print("Using MC-Dropout probabilities")
except NameError:
    probs_cal, labels_val_cf = get_probs(model, val_dl)
    probs_tst, _ = get_probs(model, test_dl)
    labels_val_cf_arr = np.array([dis_labels[i] for i in idx_val])
    print("Using standard probabilities")

labels_val_arr = np.array([dis_labels[i] for i in idx_val])

rows = []
print(f"\n{'α':>6}  {'Target':>8}  {'Coverage':>10}  {'Set Size':>10}  {'Singleton%':>11}")
print("-"*55)
for alpha in [0.05, 0.10, 0.15, 0.20]:
    n     = len(labels_val_arr)
    s     = 1.0 - probs_cal[np.arange(n), labels_val_arr]
    level = min(math.ceil((n+1)*(1-alpha)) / n, 1.0)
    q_hat = float(np.quantile(s, level))
    thr   = 1.0 - q_hat

    sets  = []
    for p in probs_tst:
        ps = list(np.where(p >= thr)[0])
        sets.append(ps if ps else [int(p.argmax())])

    coverage  = np.mean([labels_test[i] in sets[i] for i in range(len(labels_test))])
    mean_size = np.mean([len(s) for s in sets])
    sing_pct  = np.mean([len(s)==1 for s in sets]) * 100
    rows.append({"alpha":alpha,"target":1-alpha,"coverage":round(coverage,4),
                 "mean_set_size":round(mean_size,2),"singleton_pct":round(sing_pct,1)})
    print(f"  {alpha:.2f}  {1-alpha:.2f}  {coverage:10.4f}  {mean_size:10.2f}  {sing_pct:10.1f}%")

df_conf = pd.DataFrame(rows)
df_conf.to_csv(rpath("u5_conformal.csv"), index=False)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].plot(1-df_conf.alpha, df_conf.coverage, "o-", lw=2, color="#2E86C1", ms=9)
axes[0].plot([0.79,1.01],[0.79,1.01],"k--",lw=1.5)
axes[0].set_title("Coverage: Target vs Empirical\n(must be ≥ diagonal)", fontweight="bold")
axes[0].set_xlabel("Target"); axes[0].set_ylabel("Empirical"); axes[0].grid(alpha=0.3)
axes[1].plot(1-df_conf.alpha, df_conf.mean_set_size, "s-", lw=2, color="#CB4335", ms=9)
axes[1].set_title("Mean Prediction Set Size", fontweight="bold")
axes[1].set_xlabel("Target Coverage"); axes[1].grid(alpha=0.3)
axes[2].plot(1-df_conf.alpha, df_conf.singleton_pct, "^-", lw=2, color="#1E8449", ms=9)
axes[2].set_title("Singleton % (100=fully decisive)", fontweight="bold")
axes[2].set_xlabel("Target Coverage"); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(rpath("u5_conformal.png"), dpi=600, bbox_inches="tight"); plt.show()
print("\n✓ Saved: u5_conformal.png, u5_conformal.csv")

Getting val set probabilities for calibration...
Using standard probabilities

     α    Target    Coverage    Set Size   Singleton%
-------------------------------------------------------
  0.05  0.95      0.9750        1.00       100.0%
  0.10  0.90      0.9750        1.00       100.0%
  0.15  0.85      0.9750        1.00       100.0%
  0.20  0.80      0.9750        1.00       100.0%

✓ Saved: u5_conformal.png, u5_conformal.csv


## Cell 9 — V6: PCGrad Gradient Surgery
$$g_i^{PC} = g_i - \frac{g_i \cdot g_j}{\|g_j\|^2}g_j \quad \text{when } \cos(g_i,g_j) < 0$$
Removes destructive gradient interference between species and disease tasks.

In [20]:
def pcgrad_step(model, optimizer, loss_sp, loss_dis):
    """Fixed PCGrad: uses einsum instead of torch.dot for RTX 5090 compatibility."""
    optimizer.zero_grad()
    
    # Get gradients for species task
    loss_sp.backward(retain_graph=True)
    grads_sp = []
    for p in model.parameters():
        if p.grad is not None:
            grads_sp.append(p.grad.clone())
        else:
            grads_sp.append(None)
    
    optimizer.zero_grad()
    
    # Get gradients for disease task
    loss_dis.backward()
    grads_dis = []
    for p in model.parameters():
        if p.grad is not None:
            grads_dis.append(p.grad.clone())
        else:
            grads_dis.append(None)
    
    # PCGrad: resolve conflicts per parameter
    cos_sims = []
    for p, g_sp, g_dis in zip(model.parameters(), grads_sp, grads_dis):
        if g_sp is None and g_dis is None:
            continue
        elif g_sp is None:
            p.grad = g_dis.clone()
            continue
        elif g_dis is None:
            p.grad = g_sp.clone()
            continue
        
        # Flatten and cast to float32 — fixes cuBLAS CUBLAS_STATUS_NOT_SUPPORTED
        g_sp_flat  = g_sp.flatten().float()
        g_dis_flat = g_dis.flatten().float()
        
        # Use einsum instead of torch.dot — more compatible across GPU architectures
        dot          = torch.einsum('i,i->', g_sp_flat, g_dis_flat)
        norm_sp      = g_sp_flat.norm() + 1e-8
        norm_dis     = g_dis_flat.norm() + 1e-8
        cos          = (dot / (norm_sp * norm_dis)).item()
        cos_sims.append(cos)
        
        if cos < 0:
            g_sp_proj  = g_sp.float() - (dot / (norm_dis ** 2)) * g_dis.float()
            g_dis_proj = g_dis.float() - (dot / (norm_sp ** 2)) * g_sp.float()
        else:
            g_sp_proj  = g_sp.float()
            g_dis_proj = g_dis.float()
        
        # Cast back to original dtype
        p.grad = (g_sp_proj + g_dis_proj).to(p.dtype)
    
    optimizer.step()
    return float(np.mean(cos_sims)) if cos_sims else 0.0

In [21]:
# V6: PCGrad training
model_v6 = copy.deepcopy(model)
optimizer_v6 = torch.optim.AdamW(model_v6.parameters(), lr=3e-5, weight_decay=1e-4)
ce_v6 = nn.CrossEntropyLoss()

print("Training V6: PCGrad...")
cos_hist = []
best_f1_v6, patience_v6, best_wts_v6 = 0., 0, None

for ep in range(EPOCHS):
    model_v6.train()
    ep_cos, n = 0., 0
    for imgs, sp_lbs, dis_lbs in train_dl:
        imgs    = imgs.to(DEVICE)
        sp_lbs  = sp_lbs.to(DEVICE)
        dis_lbs = dis_lbs.to(DEVICE)
        sp_out, dis_out = model_v6(imgs)
        loss_sp  = ce_v6(sp_out,  sp_lbs)
        loss_dis = ce_v6(dis_out, dis_lbs) * 2.0
        cos_sim  = pcgrad_step(model_v6, optimizer_v6, loss_sp, loss_dis)
        ep_cos += cos_sim; n += 1
    mean_cos = ep_cos / n
    cos_hist.append(mean_cos)
    acc_v, f1_v, _, _ = evaluate(model_v6, val_dl)
    print(f"  Ep {ep+1:2d} | cos={mean_cos:+.4f} {'⚡PCGrad' if mean_cos<0 else '  '} | val_f1={f1_v:.4f}")
    if f1_v > best_f1_v6:
        best_f1_v6 = f1_v; patience_v6 = 0
        best_wts_v6 = copy.deepcopy(model_v6.state_dict())
    else:
        patience_v6 += 1
        if patience_v6 >= 4: print("  Early stopping"); break

model_v6.load_state_dict(best_wts_v6)
acc_v6, f1_v6, _, _ = evaluate(model_v6, test_dl)
n_conflict = sum(1 for c in cos_hist if c < 0)
print(f"\n  V6 Accuracy: {acc_v6:.4f}  Macro-F1: {f1_v6:.4f}")
print(f"  PCGrad fired in {n_conflict}/{len(cos_hist)} epochs")
all_results.append({"Model":"V6: PCGrad","Accuracy":round(acc_v6,4),"Macro-F1":round(f1_v6,4)})
torch.save(model_v6.state_dict(), rpath("model_v6.pt"))
print("✓ Done")

Training V6: PCGrad...
  Ep  1 | cos=+0.0873    | val_f1=0.9721
  Ep  2 | cos=+0.0895    | val_f1=0.9675
  Ep  3 | cos=+0.0864    | val_f1=0.9695
  Ep  4 | cos=+0.0933    | val_f1=0.9720
  Ep  5 | cos=+0.0904    | val_f1=0.9754
  Ep  6 | cos=+0.0894    | val_f1=0.9686
  Ep  7 | cos=+0.0958    | val_f1=0.9711
  Ep  8 | cos=+0.0856    | val_f1=0.9741
  Ep  9 | cos=+0.1097    | val_f1=0.9775
  Ep 10 | cos=+0.0845    | val_f1=0.9762

  V6 Accuracy: 0.9763  Macro-F1: 0.9766
  PCGrad fired in 0/10 epochs
✓ Done


In [22]:
# Fix for V7: replace BatchNorm with GroupNorm in experts (handles batch size = 1)
class SCMoEVetDermFixed(nn.Module):
    def __init__(self, n_species, n_diseases, dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            "tf_efficientnetv2_s", pretrained=pretrained,
            num_classes=0, global_pool="avg"
        )
        feat_dim = self.backbone.num_features
        self.sp_head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.GELU(), nn.BatchNorm1d(256),
            nn.Dropout(dropout), nn.Linear(256, n_species)
        )
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(feat_dim, 128), nn.GELU(),
                nn.GroupNorm(8, 128),   # ← replaces BatchNorm1d
                nn.Dropout(dropout), nn.Linear(128, n_diseases)
            ) for _ in range(n_species)
        ])

    def forward(self, x):
        feats     = self.backbone(x)
        sp_logits = self.sp_head(feats)
        k_star    = sp_logits.argmax(dim=1)
        dis_logits = torch.zeros(x.size(0), self.experts[0][-1].out_features,
                                  device=x.device)
        for k in range(len(self.experts)):
            mask = k_star == k
            if mask.sum() > 0:
                dis_logits[mask] = self.experts[k](feats[mask])
        return sp_logits, dis_logits

# Rebuild model_v7 with the fix
model_v7 = SCMoEVetDermFixed(N_SPECIES, N_DISEASES, pretrained=False).to(DEVICE)
model_v7.backbone.load_state_dict(model.backbone.state_dict())
model_v7.sp_head.load_state_dict(model.head.sp_head.state_dict())
print("✓ SCMoE fixed — ready to train")# Fix for V7: replace BatchNorm with GroupNorm in experts (handles batch size = 1)
class SCMoEVetDermFixed(nn.Module):
    def __init__(self, n_species, n_diseases, dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            "tf_efficientnetv2_s", pretrained=pretrained,
            num_classes=0, global_pool="avg"
        )
        feat_dim = self.backbone.num_features
        self.sp_head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.GELU(), nn.BatchNorm1d(256),
            nn.Dropout(dropout), nn.Linear(256, n_species)
        )
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(feat_dim, 128), nn.GELU(),
                nn.GroupNorm(8, 128),   # ← replaces BatchNorm1d
                nn.Dropout(dropout), nn.Linear(128, n_diseases)
            ) for _ in range(n_species)
        ])

    def forward(self, x):
        feats     = self.backbone(x)
        sp_logits = self.sp_head(feats)
        k_star    = sp_logits.argmax(dim=1)
        dis_logits = torch.zeros(x.size(0), self.experts[0][-1].out_features,
                                  device=x.device)
        for k in range(len(self.experts)):
            mask = k_star == k
            if mask.sum() > 0:
                dis_logits[mask] = self.experts[k](feats[mask])
        return sp_logits, dis_logits

# Rebuild model_v7 with the fix
model_v7 = SCMoEVetDermFixed(N_SPECIES, N_DISEASES, pretrained=False).to(DEVICE)
model_v7.backbone.load_state_dict(model.backbone.state_dict())
model_v7.sp_head.load_state_dict(model.head.sp_head.state_dict())
print("✓ SCMoE fixed — ready to train")

✓ SCMoE fixed — ready to train
✓ SCMoE fixed — ready to train


## Cell 10 — V7: Species-Conditioned Mixture of Experts
Hard routing: $k^* = \text{argmax}(\text{species\_logits})$ → only Expert $k^*$ activates.
One-third of disease head parameters active per sample.

In [23]:
# ── V7 Complete Fix ───────────────────────────────────────────
class SCMoEVetDermFixed(nn.Module):
    def __init__(self, n_species, n_diseases, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            "tf_efficientnetv2_s", pretrained=False,
            num_classes=0, global_pool="avg"
        )
        feat_dim = self.backbone.num_features
        self.sp_head = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.GELU(), nn.BatchNorm1d(256),
            nn.Dropout(dropout), nn.Linear(256, n_species)
        )
        # Use LayerNorm instead of BatchNorm — works with any batch size including 1
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(feat_dim, 128), nn.GELU(),
                nn.LayerNorm(128),
                nn.Dropout(dropout), nn.Linear(128, n_diseases)
            ) for _ in range(n_species)
        ])

    def forward(self, x):
        feats     = self.backbone(x)
        sp_logits = self.sp_head(feats)
        k_star    = sp_logits.argmax(dim=1)
        dis_logits = torch.zeros(x.size(0), N_DISEASES, device=x.device)
        for k in range(len(self.experts)):
            mask = k_star == k
            if mask.sum() > 0:
                dis_logits[mask] = self.experts[k](feats[mask])
        return sp_logits, dis_logits

# Build + copy backbone weights
model_v7 = SCMoEVetDermFixed(N_SPECIES, N_DISEASES).to(DEVICE)
model_v7.backbone.load_state_dict(model.backbone.state_dict())
model_v7.sp_head.load_state_dict(model.head.sp_head.state_dict())

optimizer_v7 = torch.optim.AdamW(model_v7.parameters(), lr=3e-5, weight_decay=1e-4)
ce_v7 = nn.CrossEntropyLoss()

print("Training V7: SC-MoE (fixed with LayerNorm)...")
best_f1_v7, patience_v7, best_wts_v7 = 0., 0, None

for ep in range(EPOCHS):
    model_v7.train()
    ep_loss, n = 0., 0
    for imgs, sp_lbs, dis_lbs in train_dl:
        imgs    = imgs.to(DEVICE)
        sp_lbs  = sp_lbs.to(DEVICE)
        dis_lbs = dis_lbs.to(DEVICE)
        optimizer_v7.zero_grad()
        sp_out, dis_out = model_v7(imgs)
        loss = ce_v7(sp_out, sp_lbs) + 2*ce_v7(dis_out, dis_lbs)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v7.parameters(), 1.0)
        optimizer_v7.step()
        ep_loss += loss.item(); n += 1
    acc_v, f1_v, _, _ = evaluate(model_v7, val_dl)
    print(f"  Ep {ep+1:2d} | loss={ep_loss/n:.4f} | val_f1={f1_v:.4f}")
    if f1_v > best_f1_v7:
        best_f1_v7 = f1_v; patience_v7 = 0
        best_wts_v7 = copy.deepcopy(model_v7.state_dict())
    else:
        patience_v7 += 1
        if patience_v7 >= 4: print("  Early stopping"); break

model_v7.load_state_dict(best_wts_v7)
acc_v7, f1_v7, preds_v7, _ = evaluate(model_v7, test_dl)
print(f"\n  V7 Accuracy: {acc_v7:.4f}  Macro-F1: {f1_v7:.4f}")
all_results.append({"Model":"V7: SC-MoE","Accuracy":round(acc_v7,4),"Macro-F1":round(f1_v7,4)})
torch.save(model_v7.state_dict(), rpath("model_v7.pt"))
print("✓ Done")

Training V7: SC-MoE (fixed with LayerNorm)...
  Ep  1 | loss=1.1669 | val_f1=0.9685
  Ep  2 | loss=0.2260 | val_f1=0.9681
  Ep  3 | loss=0.1409 | val_f1=0.9734
  Ep  4 | loss=0.1087 | val_f1=0.9681
  Ep  5 | loss=0.0848 | val_f1=0.9707
  Ep  6 | loss=0.0999 | val_f1=0.9791
  Ep  7 | loss=0.0830 | val_f1=0.9760
  Ep  8 | loss=0.0725 | val_f1=0.9759
  Ep  9 | loss=0.0533 | val_f1=0.9812
  Ep 10 | loss=0.0676 | val_f1=0.9778

  V7 Accuracy: 0.9776  Macro-F1: 0.9777
✓ Done


## Cell 11 — V8: SSM Complexity Benchmark
No training required.

In [24]:
H, W, d = 10, 10, 256
n = H*W
mha_flops = n*n*d*2
ssm_flops = n*d
print(f"MHA FLOPs: {mha_flops:,}  O(n²·d)")
print(f"SSM FLOPs: {ssm_flops:,}  O(n·d)")
print(f"Theoretical speedup: {mha_flops//ssm_flops}×")

# Actual timing
x_bench = torch.randn(8, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
model.eval()
with torch.no_grad():
    for _ in range(5): model(x_bench)   # warmup
start = time.time()
with torch.no_grad():
    for _ in range(100): model(x_bench)
t_ms = (time.time()-start)/100*1000
print(f"\nBaseline inference (batch=8): {t_ms:.2f} ms")
print(f"Per image: {t_ms/8:.2f} ms")
print(f"Throughput: {8/(t_ms/1000):.0f} images/sec")
pd.DataFrame([{"mha_flops":mha_flops,"ssm_flops":ssm_flops,
               "speedup":mha_flops//ssm_flops,
               "ms_per_batch_8":round(t_ms,2),
               "imgs_per_sec":round(8/(t_ms/1000),1)}
              ]).to_csv(rpath("v8_benchmark.csv"), index=False)
print("\n✓ Saved: v8_benchmark.csv")

MHA FLOPs: 5,120,000  O(n²·d)
SSM FLOPs: 25,600  O(n·d)
Theoretical speedup: 200×

Baseline inference (batch=8): 22.20 ms
Per image: 2.77 ms
Throughput: 360 images/sec

✓ Saved: v8_benchmark.csv


## Cell 12 — Final Ablation Table + Chart
Run after all upgrade cells.

In [25]:
print("█"*55)
print("  FINAL ABLATION RESULTS — PyTorch")
print("█"*55)

if not all_results:
    print("No results yet — run Cells 3-11 first")
else:
    df = pd.DataFrame(all_results)
    df["ΔAcc vs Baseline"] = (df["Accuracy"] - df["Accuracy"].iloc[0]).round(4)
    df["ΔF1 vs Baseline"]  = (df["Macro-F1"] - df["Macro-F1"].iloc[0]).round(4)
    print(df.to_string(index=False))
    df.to_csv(rpath("final_ablation.csv"), index=False)

    if len(df) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(16,5))
        names = df["Model"].tolist()
        accs  = df["Accuracy"].tolist()
        f1s   = df["Macro-F1"].tolist()
        pal   = (["#BDC3C7"] +
                 ["#2E86C1","#CB4335","#1E8449","#8E44AD",
                  "#D4AC0D","#7D3C98","#1A5276","#A93226"][:len(df)-1])
        x = np.arange(len(df))
        for ax, vals, ylabel, title in [
            (axes[0], accs, "Accuracy", "Disease Accuracy"),
            (axes[1], f1s,  "Macro-F1", "Macro-F1"),
        ]:
            ax.bar(x, vals, color=pal[:len(x)])
            ax.set_xticks(x)
            ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
            ax.set_ylabel(ylabel)
            ax.set_title(f"{title} — All 8 Upgrades", fontweight="bold")
            ax.set_ylim(max(0,min(vals)-0.06), min(1.0,max(vals)+0.06))
            ax.grid(axis="y", alpha=0.3)
            for i,v in enumerate(vals):
                ax.text(i, v+0.003, f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")
        plt.suptitle("SGCA-VetDerm++ PyTorch | Complete Upgrade Ablation",
                      fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(rpath("final_ablation.png"), dpi=600, bbox_inches="tight")
        plt.show()

    print(f"\n✓ All results saved to ./{RESULTS_DIR}/")
    for f in sorted(os.listdir(RESULTS_DIR)):
        size = os.path.getsize(os.path.join(RESULTS_DIR, f))
        print(f"  {f:<45} {size/1024:.1f} KB")

███████████████████████████████████████████████████████
  FINAL ABLATION RESULTS — PyTorch
███████████████████████████████████████████████████████
              Model  Accuracy  Macro-F1  ΔAcc vs Baseline  ΔF1 vs Baseline
      SGCA Baseline    0.0500    0.0165            0.0000           0.0000
    U1: Kendall-Gal    0.9763    0.9763            0.9263           0.9598
U2: MC-Dropout T=30    0.9526    0.9530            0.9026           0.9365
   U3: Prototypical    0.9743    0.9742            0.9243           0.9577
       U4: Focal+LS    0.9711    0.9710            0.9211           0.9545
         V6: PCGrad    0.9763    0.9766            0.9263           0.9601
         V7: SC-MoE    0.9776    0.9777            0.9276           0.9612

✓ All results saved to ./sgca_results_pt/
  combined_learning_curve.png                   347.7 KB
  confusion_matrix.png                          5236.4 KB
  cv_results.csv                                0.2 KB
  external_validation.csv               

In [26]:
# ── Clean duplicate U2 entry first ───────────────────────────
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches

df_plot = pd.DataFrame([r for r in all_results 
                        if r["Model"] != "SGCA Baseline"])
baseline_f1  = 0.0165
baseline_acc = 0.0500

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor("white")

names  = df_plot["Model"].tolist()
accs   = df_plot["Accuracy"].tolist()
f1s    = df_plot["Macro-F1"].tolist()
colors = ["#2E86C1","#CB4335","#1E8449","#8E44AD","#D4AC0D","#7D3C98","#1A5276"]
x = np.arange(len(df_plot))

for ax, vals, ylabel, title in [
    (axes[0], accs, "Accuracy",  "Disease Accuracy — All Upgrades"),
    (axes[1], f1s,  "Macro-F1",  "Macro-F1 — All Upgrades")
]:
    bars = ax.bar(x, vals, color=colors[:len(x)], width=0.6,
                  edgecolor="white", linewidth=0.8)
    
    # Baseline reference line
    baseline_val = baseline_acc if ylabel == "Accuracy" else baseline_f1
    ax.axhline(baseline_val, color="gray", ls="--", lw=1.2, alpha=0.6,
               label=f"ImageNet baseline ({baseline_val:.3f})")
    
    # Y-axis: zoom in to meaningful range
    y_min = min(vals) - 0.015
    y_max = min(1.0, max(vals) + 0.025)
    ax.set_ylim(y_min, y_max)
    
    # Value labels — inside bars near top, never clipped
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() - 0.006,
                f"{v:.4f}",
                ha="center", va="top",
                fontsize=9.5, fontweight="bold", color="white")
    
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=35, ha="right", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.legend(fontsize=9, framealpha=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "SGCA-VetDerm++ | Upgrade Ablation Study\n"
    f"Best: Combined U1+U3+U4 → Macro-F1: 0.9821 ± 0.0060 (5-Fold CV)",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(rpath("final_ablation_clean.png"), dpi=600,
            bbox_inches="tight", facecolor="white")
plt.show()
print("✓ Saved: final_ablation_clean.png")

✓ Saved: final_ablation_clean.png


In [27]:
# ══════════════════════════════════════════════════════
# COMBINED MODEL: U1 + U3 + U4 + Cosine LR + 30 Epochs
# ══════════════════════════════════════════════════════

COMBINED_EPOCHS = 30

class CombinedLoss(nn.Module):
    """U1 (Kendall-Gal) + U3 (Prototypical) + U4 (Focal+LS) combined."""
    def __init__(self, n_diseases, gamma=2.0, epsilon=0.1, proto_lam=0.1):
        super().__init__()
        self.K          = n_diseases
        self.gamma      = gamma
        self.eps        = epsilon
        self.proto_lam  = proto_lam
        self.ce         = nn.CrossEntropyLoss()
        # U1: learnable log-variance scalars
        self.log_s_sp   = nn.Parameter(torch.zeros(1))
        self.log_s_dis  = nn.Parameter(torch.zeros(1))

    def focal_ls(self, logits, targets):
        """U4: Focal + Label Smoothing loss."""
        probs  = F.softmax(logits, dim=1)
        y_oh   = F.one_hot(targets, self.K).float()
        q      = (1-self.eps)*y_oh + self.eps/self.K
        p_true = (probs * y_oh).sum(dim=1)
        focal_w = (1.0 - p_true).pow(self.gamma)
        log_p  = torch.log(probs + 1e-8)
        ls_ce  = -(q * log_p).sum(dim=1)
        return (focal_w * ls_ce).mean()

    def proto(self, emb, y_sp, y_dis, margin=1.0):
        """U3: Hierarchical prototypical loss."""
        pulls, pushes = [], []
        for sp in range(N_SPECIES):
            for dis in range(N_DISEASES):
                mask = (y_sp == sp) & (y_dis == dis)
                if mask.sum() < 2: continue
                z = emb[mask]
                proto = z.mean(dim=0, keepdim=True)
                pulls.append(((z - proto)**2).sum(dim=1).mean())
        for grp in CROSS_GROUPS:
            protos = []
            for d in grp:
                mask = y_dis == d
                if mask.sum() < 1: continue
                protos.append(emb[mask].mean(dim=0))
            for i in range(len(protos)):
                for j in range(i+1, len(protos)):
                    d2 = ((protos[i] - protos[j])**2).sum()
                    pushes.append(F.relu(margin - d2))
        L_pull = torch.stack(pulls).mean()  if pulls  else torch.tensor(0., device=emb.device)
        L_push = torch.stack(pushes).mean() if pushes else torch.tensor(0., device=emb.device)
        return 0.5*L_pull + 0.5*L_push

    def forward(self, sp_out, dis_out, sp_lbs, dis_lbs, emb):
        L_sp  = self.ce(sp_out, sp_lbs)
        L_dis = self.focal_ls(dis_out, dis_lbs)          # U4
        # U1: uncertainty weighting
        loss  = (torch.exp(-self.log_s_sp)  * L_sp  + self.log_s_sp +
                 torch.exp(-self.log_s_dis) * L_dis + self.log_s_dis)
        # U3: prototypical
        loss  = loss + self.proto_lam * self.proto(emb, sp_lbs, dis_lbs)
        return loss


# ── Build model ───────────────────────────────────────
print("Building combined model (U1+U3+U4)...")
model_combined = SGCAVetDerm(N_SPECIES, N_DISEASES, pretrained=True).to(DEVICE)
criterion      = CombinedLoss(N_DISEASES).to(DEVICE)

optimizer_comb = torch.optim.AdamW(
    list(model_combined.parameters()) + list(criterion.parameters()),
    lr=3e-4, weight_decay=1e-4
)
# Cosine LR schedule — smoothly decays over 30 epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_comb, T_max=COMBINED_EPOCHS, eta_min=1e-6
)

# ── Training loop ─────────────────────────────────────
best_f1_c, patience_c, best_wts_c = 0., 0, None
history = []
print(f"Training for up to {COMBINED_EPOCHS} epochs...")
print(f"{'Ep':>3} {'Loss':>8} {'σ_sp':>7} {'σ_dis':>7} {'LR':>9} {'val_F1':>8}")
print("─"*50)

for ep in range(COMBINED_EPOCHS):
    model_combined.train()
    ep_loss, n = 0., 0
    for imgs, sp_lbs, dis_lbs in train_dl:
        imgs    = imgs.to(DEVICE)
        sp_lbs  = sp_lbs.to(DEVICE)
        dis_lbs = dis_lbs.to(DEVICE)
        optimizer_comb.zero_grad()
        sp_out, dis_out, emb = model_combined(imgs, return_embedding=True)
        loss = criterion(sp_out, dis_out, sp_lbs, dis_lbs, emb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_combined.parameters(), 1.0)
        optimizer_comb.step()
        ep_loss += loss.item(); n += 1
    scheduler.step()

    acc_v, f1_v, _, _ = evaluate(model_combined, val_dl)
    lr_now   = scheduler.get_last_lr()[0]
    sig_sp   = float(torch.exp(criterion.log_s_sp/2).detach().cpu())
    sig_dis  = float(torch.exp(criterion.log_s_dis/2).detach().cpu())
    history.append({"ep":ep+1,"loss":ep_loss/n,"f1":f1_v,"lr":lr_now})

    print(f"{ep+1:>3} {ep_loss/n:>8.4f} {sig_sp:>7.4f} {sig_dis:>7.4f} {lr_now:>9.2e} {f1_v:>8.4f}")

    if f1_v > best_f1_c:
        best_f1_c = f1_v; patience_c = 0
        best_wts_c = copy.deepcopy(model_combined.state_dict())
        torch.save(best_wts_c, rpath("model_combined_best.pt"))
    else:
        patience_c += 1
        if patience_c >= 7:
            print(f"  Early stopping at ep {ep+1}")
            break

# ── Evaluate ──────────────────────────────────────────
model_combined.load_state_dict(best_wts_c)
acc_c, f1_c, preds_c, _ = evaluate(model_combined, test_dl)
print(f"\n{'='*50}")
print(f"  COMBINED MODEL RESULT")
print(f"{'='*50}")
print(f"  Accuracy : {acc_c:.4f}")
print(f"  Macro-F1 : {f1_c:.4f}")
print(f"  Best val  : {best_f1_c:.4f}")
print(f"{'='*50}")
all_results.append({"Model":"Combined U1+U3+U4","Accuracy":round(acc_c,4),"Macro-F1":round(f1_c,4)})

# ── Learning curve ────────────────────────────────────
df_hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].plot(df_hist.ep, df_hist.f1,   lw=2, color="#2E86C1")
axes[0].axhline(best_f1_c, color="red", ls="--", lw=1.5, label=f"Best={best_f1_c:.4f}")
axes[0].set_title("Combined Model — Val Macro-F1", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(df_hist.ep, df_hist.loss, lw=2, color="#CB4335")
axes[1].set_title("Combined Model — Training Loss", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(rpath("combined_learning_curve.png"), dpi=600, bbox_inches="tight")
plt.show()
print(f"\n✓ Saved: model_combined_best.pt, combined_learning_curve.png")

Building combined model (U1+U3+U4)...


Training for up to 30 epochs...
 Ep     Loss    σ_sp   σ_dis        LR   val_F1
──────────────────────────────────────────────────
  1   1.3995  0.9344  0.9873  2.99e-04   0.8605
  2   0.3204  0.8707  0.9204  2.97e-04   0.9081
  3  -0.0659  0.8143  0.8570  2.93e-04   0.9075
  4  -0.4227  0.7631  0.7997  2.87e-04   0.9381
  5  -0.7604  0.7156  0.7478  2.80e-04   0.9397
  6  -0.9721  0.6736  0.7032  2.71e-04   0.9355
  7  -1.2213  0.6359  0.6628  2.62e-04   0.9633
  8  -1.4836  0.6017  0.6268  2.51e-04   0.9532
  9  -1.6424  0.5721  0.5956  2.38e-04   0.9579
 10  -1.9306  0.5442  0.5663  2.25e-04   0.9605
 11  -2.1613  0.5188  0.5397  2.11e-04   0.9639
 12  -2.3757  0.4964  0.5152  1.97e-04   0.9581
 13  -2.5462  0.4769  0.4941  1.82e-04   0.9578
 14  -2.7528  0.4591  0.4752  1.66e-04   0.9683
 15  -2.9070  0.4435  0.4591  1.50e-04   0.9762
 16  -3.0949  0.4294  0.4445  1.35e-04   0.9697
 17  -3.2612  0.4171  0.4316  1.19e-04   0.9715
 18  -3.3544  0.4067  0.4207  1.04e-04   0.9691
 19  

In [28]:
# TTA: average predictions over 5 augmented versions of each test image
tta_transforms = [
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(),
               T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomHorizontalFlip(p=1.0),
               T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomVerticalFlip(p=1.0),
               T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((int(IMG_SIZE*1.1),int(IMG_SIZE*1.1))),
               T.CenterCrop(IMG_SIZE), T.ToTensor(),
               T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ColorJitter(brightness=0.2,contrast=0.2),
               T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
]

@torch.no_grad()
def tta_predict(model, indices, transforms, device=DEVICE):
    model.eval()
    all_probs = []
    for tf in transforms:
        ds  = VetDermDataset(indices, tf)
        dl  = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=True)
        probs_t = []
        for imgs, _, _ in dl:
            _, dis_out = model(imgs.to(device))
            probs_t.append(F.softmax(dis_out, dim=1).cpu().numpy())
        all_probs.append(np.concatenate(probs_t))
    avg_probs = np.mean(all_probs, axis=0)
    return avg_probs.argmax(axis=1)

print("Running TTA (5 augmentations)...")
preds_tta = tta_predict(model_combined, idx_te, tta_transforms)
acc_tta = accuracy_score(labels_test, preds_tta)
f1_tta  = f1_score(labels_test, preds_tta, average="macro", zero_division=0)
print(f"\n  TTA Accuracy : {acc_tta:.4f}")
print(f"  TTA Macro-F1 : {f1_tta:.4f}")
print(f"  vs no TTA    : {f1_c:.4f}")
print(f"  TTA gain     : +{f1_tta - f1_c:.4f}")
all_results.append({"Model":"Combined+TTA","Accuracy":round(acc_tta,4),"Macro-F1":round(f1_tta,4)})

Running TTA (5 augmentations)...

  TTA Accuracy : 0.9763
  TTA Macro-F1 : 0.9766
  vs no TTA    : 0.9753
  TTA gain     : +0.0013


In [29]:
# ══════════════════════════════════════════════════════
# 5-FOLD CROSS VALIDATION on Combined Model
# ══════════════════════════════════════════════════════
from sklearn.model_selection import StratifiedKFold

CV_EPOCHS = 20
N_FOLDS   = 5

skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_idx = np.concatenate([idx_tr, idx_val, idx_te])  # use full dataset for CV
all_dis = dis_labels[all_idx]

fold_results = []
print(f"Running {N_FOLDS}-Fold Cross Validation ({CV_EPOCHS} epochs/fold)...")
print(f"Estimated time: ~{N_FOLDS * CV_EPOCHS * 2} minutes on RTX 5090\n")

for fold, (tr_idx, val_idx) in enumerate(skf.split(all_idx, all_dis)):
    print(f"{'='*50}")
    print(f"  FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")

    fold_tr  = all_idx[tr_idx]
    fold_val = all_idx[val_idx]

    fold_tr_ds  = VetDermDataset(fold_tr,  train_tf)
    fold_val_ds = VetDermDataset(fold_val, val_tf)
    fold_tr_dl  = DataLoader(fold_tr_ds,  batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True)
    fold_val_dl = DataLoader(fold_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    # Fresh model + combined loss for each fold
    fold_model = SGCAVetDerm(N_SPECIES, N_DISEASES, pretrained=True).to(DEVICE)
    fold_crit  = CombinedLoss(N_DISEASES).to(DEVICE)
    fold_opt   = torch.optim.AdamW(
        list(fold_model.parameters()) + list(fold_crit.parameters()),
        lr=1e-4, weight_decay=1e-4
    )
    fold_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        fold_opt, T_max=CV_EPOCHS, eta_min=1e-6
    )

    best_f1_fold, best_acc_fold = 0., 0.
    patience_fold = 0

    for ep in range(CV_EPOCHS):
        fold_model.train()
        for imgs, sp_lbs, dis_lbs in fold_tr_dl:
            imgs    = imgs.to(DEVICE)
            sp_lbs  = sp_lbs.to(DEVICE)
            dis_lbs = dis_lbs.to(DEVICE)
            fold_opt.zero_grad()
            sp_out, dis_out, emb = fold_model(imgs, return_embedding=True)
            loss = fold_crit(sp_out, dis_out, sp_lbs, dis_lbs, emb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fold_model.parameters(), 1.0)
            fold_opt.step()
        fold_sched.step()

        acc_f, f1_f, _, _ = evaluate(fold_model, fold_val_dl)
        print(f"  Ep {ep+1:2d} | val_f1={f1_f:.4f}")
        if f1_f > best_f1_fold:
            best_f1_fold  = f1_f
            best_acc_fold = acc_f
            patience_fold = 0
        else:
            patience_fold += 1
            if patience_fold >= 5:
                print(f"  Early stopping"); break

    fold_results.append({"fold":fold+1,"accuracy":best_acc_fold,"macro_f1":best_f1_fold})
    print(f"\n  Fold {fold+1} best — Acc: {best_acc_fold:.4f}  F1: {best_f1_fold:.4f}\n")
    del fold_model, fold_crit, fold_opt
    torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────
df_cv = pd.DataFrame(fold_results)
mean_f1  = df_cv.macro_f1.mean()
std_f1   = df_cv.macro_f1.std()
mean_acc = df_cv.accuracy.mean()
std_acc  = df_cv.accuracy.std()

print(f"\n{'█'*50}")
print(f"  5-FOLD CV FINAL RESULTS")
print(f"{'█'*50}")
print(df_cv.to_string(index=False))
print(f"\n  Macro-F1 : {mean_f1:.4f} ± {std_f1:.4f}")
print(f"  Accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
print(f"{'█'*50}")
df_cv.to_csv(rpath("cv_results.csv"), index=False)
print(f"\n✓ Saved: cv_results.csv")

Running 5-Fold Cross Validation (20 epochs/fold)...
Estimated time: ~200 minutes on RTX 5090

  FOLD 1/5
  Ep  1 | val_f1=0.8665
  Ep  2 | val_f1=0.9428
  Ep  3 | val_f1=0.9638
  Ep  4 | val_f1=0.9730
  Ep  5 | val_f1=0.9766
  Ep  6 | val_f1=0.9786
  Ep  7 | val_f1=0.9810
  Ep  8 | val_f1=0.9842
  Ep  9 | val_f1=0.9815
  Ep 10 | val_f1=0.9838
  Ep 11 | val_f1=0.9856
  Ep 12 | val_f1=0.9869
  Ep 13 | val_f1=0.9875
  Ep 14 | val_f1=0.9877
  Ep 15 | val_f1=0.9828
  Ep 16 | val_f1=0.9869
  Ep 17 | val_f1=0.9869
  Ep 18 | val_f1=0.9868
  Ep 19 | val_f1=0.9863
  Early stopping

  Fold 1 best — Acc: 0.9877  F1: 0.9877

  FOLD 2/5
  Ep  1 | val_f1=0.8475
  Ep  2 | val_f1=0.9347
  Ep  3 | val_f1=0.9571
  Ep  4 | val_f1=0.9498
  Ep  5 | val_f1=0.9601
  Ep  6 | val_f1=0.9644
  Ep  7 | val_f1=0.9662
  Ep  8 | val_f1=0.9748
  Ep  9 | val_f1=0.9698
  Ep 10 | val_f1=0.9736
  Ep 11 | val_f1=0.9725
  Ep 12 | val_f1=0.9788
  Ep 13 | val_f1=0.9759
  Ep 14 | val_f1=0.9757
  Ep 15 | val_f1=0.9771
  Ep 16 |

In [30]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model_combined.load_state_dict(torch.load(rpath("model_combined_best.pt")))
acc_final, f1_final, preds_final, labels_final = evaluate(model_combined, test_dl)

cm = confusion_matrix(labels_final, preds_final)
fig, ax = plt.subplots(figsize=(16, 14))
disp = ConfusionMatrixDisplay(cm, display_labels=DISEASE_NAMES)
disp.plot(ax=ax, xticks_rotation=60, colorbar=False, cmap="YlOrBr")
ax.set_title(f"SGCA-VetDerm++ Confusion Matrix\nMacro-F1: {f1_final:.4f}", 
             fontweight="bold", fontsize=18)
plt.tight_layout()
plt.savefig(rpath("confusion_matrix.png"), dpi=600, bbox_inches="tight")
plt.show()
print("✓ Saved: confusion_matrix.png")

✓ Saved: confusion_matrix.png


In [31]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model_combined.load_state_dict(torch.load(rpath("model_combined_best.pt"), map_location=DEVICE))
acc_final, f1_final, preds_final, labels_final = evaluate(model_combined, test_dl)

cm = confusion_matrix(labels_final, preds_final)

fig, ax = plt.subplots(figsize=(22, 19))

disp = ConfusionMatrixDisplay(cm, display_labels=[d.replace("_"," ").title() for d in DISEASE_NAMES])
disp.plot(ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")

# ── Numbers inside cells ──────────────────────────────────────
for text in disp.text_.ravel():
    text.set_fontsize(16)  
    fontweight="bold"     # ← numbers inside CM

# ── Disease name labels ───────────────────────────────────────
ax.set_xticklabels(ax.get_xticklabels(), fontsize=16, fontweight="bold", ha="right")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=16, fontweight="bold")

# ── Axis titles ───────────────────────────────────────────────
ax.set_xlabel("Predicted Label", fontsize=14, fontweight="bold", labelpad=12)
ax.set_ylabel("True Label",      fontsize=14, fontweight="bold", labelpad=12)
ax.set_title(
    f"SGCA-VetDerm++ Confusion Matrix\nMacro-F1: {f1_final:.4f}",
    fontsize=14, fontweight="bold", pad=18
)

plt.tight_layout()
plt.savefig(rpath("confusion_matrix.png"), dpi=900, bbox_inches="tight")
plt.show()
print("✓ Saved")

✓ Saved


In [32]:
# SGCA Baseline — no upgrades, just the base architecture
# This was already run as "SGCA Baseline" in your upgrade cells
# Check your ablation CSV

import pandas as pd
df = pd.read_csv(rpath("final_ablation.csv"))
print(df.to_string())

                 Model  Accuracy  Macro-F1  ΔAcc vs Baseline  ΔF1 vs Baseline
0        SGCA Baseline    0.0500    0.0165            0.0000           0.0000
1      U1: Kendall-Gal    0.9763    0.9763            0.9263           0.9598
2  U2: MC-Dropout T=30    0.9526    0.9530            0.9026           0.9365
3     U3: Prototypical    0.9743    0.9742            0.9243           0.9577
4         U4: Focal+LS    0.9711    0.9710            0.9211           0.9545
5           V6: PCGrad    0.9763    0.9766            0.9263           0.9601
6           V7: SC-MoE    0.9776    0.9777            0.9276           0.9612
